In [ ]:
# Step 1: load the final analytical corpus and final institution dictionary

from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
OUTPUT_DIR = PROJECT_ROOT / "outputs"

CORPUS_PATH = OUTPUT_DIR / "final_analysis_corpus_v1.csv"
DICTIONARY_PATH = OUTPUT_DIR / "institution_dictionary_candidate_final_v1.xlsx"

final_corpus = pd.read_csv(CORPUS_PATH)
dictionary = pd.read_excel(DICTIONARY_PATH)

print("Final corpus shape:", final_corpus.shape)
print("Dictionary shape:", dictionary.shape)

print("\nCorpus columns:")
print(final_corpus.columns.tolist())

print("\nDictionary columns:")
print(dictionary.columns.tolist())

In [ ]:
# Step 2: validate final analytical inputs

# Final corpus checks
print("Unique transcript IDs:",
      final_corpus["participant_ids_str"].nunique())

print("Missing analysis text:",
      final_corpus["analysis_text_final"].isna().sum())

print("Empty analysis text:",
      final_corpus["analysis_text_final"]
      .fillna("")
      .str.strip()
      .eq("")
      .sum())

# Dictionary checks
dictionary["include/exclude"] = (
    dictionary["include/exclude"]
    .astype(str)
    .str.strip()
    .str.lower()
)

included_dictionary = dictionary[
    dictionary["include/exclude"] == "include"
].copy()

print("\nIncluded dictionary rows:", len(included_dictionary))

print("\nIncluded categories:")
print(
    included_dictionary["category"]
    .value_counts()
)

print("\nMissing keywords:",
      included_dictionary["keyword"].isna().sum())

print(
    "Duplicate category-keyword rows:",
    included_dictionary
    .duplicated(["category", "keyword"])
    .sum()
)

In [ ]:
# Step 3: extract full-corpus KWIC records using the final dictionary

import re

WINDOW_WORDS = 50

def extract_word_window(text, match_start, match_end, window_words=50):
    """
    Extract up to `window_words` words before and after a keyword match.
    """
    text = str(text)

    left_text = text[:match_start]
    matched_text = text[match_start:match_end]
    right_text = text[match_end:]

    left_context = " ".join(left_text.split()[-window_words:])
    right_context = " ".join(right_text.split()[:window_words])

    context = (
        left_context
        + " "
        + matched_text
        + " "
        + right_context
    ).strip()

    return left_context, matched_text, right_context, context


kwic_rows = []

for _, corpus_row in final_corpus.iterrows():

    transcript_id = corpus_row["participant_ids_str"]
    file_name = corpus_row["file_name"]
    text = str(corpus_row["analysis_text_final"])

    for _, dict_row in included_dictionary.iterrows():

        category = dict_row["category"]
        keyword = str(dict_row["keyword"]).strip()

        # Whole-word / phrase matching, case-insensitive
        pattern = re.compile(
            rf"(?<!\w){re.escape(keyword)}(?!\w)",
            flags=re.IGNORECASE
        )

        for match in pattern.finditer(text):

            left_context, matched_text, right_context, context = (
                extract_word_window(
                    text,
                    match.start(),
                    match.end(),
                    window_words=WINDOW_WORDS
                )
            )

            kwic_rows.append({
                "transcript_id": transcript_id,
                "file_name": file_name,
                "category": category,
                "keyword": keyword,
                "matched_text": matched_text,
                "context": context,
                "left_context": left_context,
                "right_context": right_context,
                "match_start": match.start(),
                "match_end": match.end(),
                "partial_transcript": corpus_row["partial_transcript"],
                "word_count": corpus_row["word_count"]
            })

full_kwic = pd.DataFrame(kwic_rows)

print("Full KWIC rows:", len(full_kwic))
print(
    "Unique transcripts with at least one hit:",
    full_kwic["transcript_id"].nunique()
)

display(full_kwic.head(10))

In [ ]:
# Step 4: summarise full-corpus KWIC retrieval

# Category-level summary
category_summary = (
    full_kwic
    .groupby("category")
    .agg(
        kwic_hits=("keyword", "size"),
        unique_transcripts=("transcript_id", "nunique")
    )
    .reset_index()
    .sort_values("kwic_hits", ascending=False)
)

print("Category-level retrieval summary:")
display(category_summary)


# Keyword-level summary
keyword_summary = (
    full_kwic
    .groupby(["category", "keyword"])
    .agg(
        kwic_hits=("keyword", "size"),
        unique_transcripts=("transcript_id", "nunique")
    )
    .reset_index()
    .sort_values(
        ["category", "kwic_hits"],
        ascending=[True, False]
    )
)

print("\nKeyword-level retrieval summary:")
display(keyword_summary)

In [ ]:
# Step 4.1: export full-corpus KWIC results

FULL_KWIC_PATH = OUTPUT_DIR / "institution_kwic_results_v2.csv"

full_kwic.to_csv(
    FULL_KWIC_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("Full-corpus KWIC exported successfully.")
print("Rows:", len(full_kwic))
print("Saved to:", FULL_KWIC_PATH)

### Retrieval summary and validation rationale

The final dictionary retrieved 8,865 KWIC records across all 141 transcripts. Police and legal institutions account for the largest number of hits, while health and welfare/state categories also show substantial corpus coverage.

However, initial validation work indicated lower validity for health and welfare/state, and several high-frequency terms in these categories may be contextually ambiguous. A balanced validation sample is therefore drawn from all five categories before TF-IDF and NMF analysis, in order to assess retrieval precision at full-corpus scale.

In [ ]:
# Step 5: draw a balanced full-corpus validation sample

VALIDATION_TARGETS = {
    "police": 50,
    "legal": 50,
    "health": 50,
    "welfare_state": 50,
    "support_sector": 50
}

validation_parts = []

for category, target_n in VALIDATION_TARGETS.items():
    
    category_df = full_kwic[
        full_kwic["category"] == category
    ].copy()
    
    sampled = category_df.sample(
        n=min(target_n, len(category_df)),
        random_state=42
    )
    
    validation_parts.append(sampled)

validation_sample = pd.concat(
    validation_parts,
    ignore_index=True
)

print("Total validation sample:", len(validation_sample))

print("\nSample by category:")
print(
    validation_sample["category"]
    .value_counts()
)

print("\nUnique transcripts represented by category:")
print(
    validation_sample
    .groupby("category")["transcript_id"]
    .nunique()
)

In [ ]:
# Step 6: inspect keyword coverage in the validation sample

sample_keyword_summary = (
    validation_sample
    .groupby(["category", "keyword"])
    .agg(
        sample_n=("keyword", "size"),
        unique_transcripts=("transcript_id", "nunique")
    )
    .reset_index()
    .sort_values(
        ["category", "sample_n"],
        ascending=[True, False]
    )
)

display(sample_keyword_summary)

print("\nNumber of keywords represented by category:")
print(
    validation_sample
    .groupby("category")["keyword"]
    .nunique()
)

In [ ]:
# Step 6.1: identify included dictionary keywords missing from the validation sample

all_keywords = (
    included_dictionary[
        ["category", "keyword"]
    ]
    .drop_duplicates()
)

sampled_keywords = (
    validation_sample[
        ["category", "keyword"]
    ]
    .drop_duplicates()
)

missing_sample_keywords = (
    all_keywords
    .merge(
        sampled_keywords,
        on=["category", "keyword"],
        how="left",
        indicator=True
    )
)

missing_sample_keywords = missing_sample_keywords[
    missing_sample_keywords["_merge"] == "left_only"
][["category", "keyword"]]

print("Keywords not represented in the validation sample:")
display(missing_sample_keywords)

print("\nNumber missing by category:")
print(
    missing_sample_keywords["category"]
    .value_counts()
)

In [ ]:
# Step 6.2: check full-corpus availability of keywords missing from validation sample

missing_keyword_availability = (
    missing_sample_keywords
    .merge(
        keyword_summary[
            ["category", "keyword", "kwic_hits", "unique_transcripts"]
        ],
        on=["category", "keyword"],
        how="left"
    )
)

missing_keyword_availability["kwic_hits"] = (
    missing_keyword_availability["kwic_hits"]
    .fillna(0)
    .astype(int)
)

missing_keyword_availability["unique_transcripts"] = (
    missing_keyword_availability["unique_transcripts"]
    .fillna(0)
    .astype(int)
)

display(missing_keyword_availability)

In [ ]:
# Step 6.3: create supplementary coverage sample for uncovered keywords

supplement_parts = []

for _, row in missing_keyword_availability.iterrows():

    category = row["category"]
    keyword = row["keyword"]
    available_n = int(row["kwic_hits"])

    # No full-corpus hits: record only, no supplement
    if available_n == 0:
        continue

    keyword_pool = full_kwic[
        (full_kwic["category"] == category) &
        (full_kwic["keyword"] == keyword)
    ].copy()

    # Remove records already included in the main validation sample
    existing_keys = set(
        zip(
            validation_sample["transcript_id"],
            validation_sample["match_start"],
            validation_sample["match_end"]
        )
    )

    keyword_pool = keyword_pool[
        ~keyword_pool.apply(
            lambda row: (
                row["transcript_id"],
                row["match_start"],
                row["match_end"]
            ) in existing_keys,
            axis=1
        )
    ].copy()

    # Supplement rule:
    # <=5 full-corpus hits: review all available records
    # >5 hits: randomly review 3
    if available_n <= 5:
        supplement = keyword_pool.copy()
    else:
        supplement = keyword_pool.sample(
            n=min(3, len(keyword_pool)),
            random_state=42
        )

    supplement["sample_source"] = "keyword_supplement"
    supplement_parts.append(supplement)


# Mark the original category-balanced random sample
validation_sample["sample_source"] = "category_random"

keyword_supplement = pd.concat(
    supplement_parts,
    ignore_index=True
)

validation_sample_final = pd.concat(
    [validation_sample, keyword_supplement],
    ignore_index=True
)

print("Main category-random sample:", len(validation_sample))
print("Keyword coverage supplement:", len(keyword_supplement))
print("Final validation sample:", len(validation_sample_final))

print("\nSupplement by category and keyword:")
display(
    keyword_supplement
    .groupby(["category", "keyword"])
    .size()
    .reset_index(name="supplement_n")
)

In [ ]:
# Step 6.4: final keyword coverage check

final_sampled_keywords = (
    validation_sample_final[
        ["category", "keyword"]
    ]
    .drop_duplicates()
)

final_keyword_coverage = (
    included_dictionary[
        ["category", "keyword"]
    ]
    .drop_duplicates()
    .merge(
        keyword_summary[
            ["category", "keyword", "kwic_hits"]
        ],
        on=["category", "keyword"],
        how="left"
    )
)

final_keyword_coverage["kwic_hits"] = (
    final_keyword_coverage["kwic_hits"]
    .fillna(0)
    .astype(int)
)

final_keyword_coverage = final_keyword_coverage.merge(
    final_sampled_keywords.assign(in_validation_sample=True),
    on=["category", "keyword"],
    how="left"
)

final_keyword_coverage["in_validation_sample"] = (
    final_keyword_coverage["in_validation_sample"]
    .fillna(False)
)

display(final_keyword_coverage)

print("\nIncluded dictionary terms:", len(final_keyword_coverage))

print(
    "Terms with full-corpus hits:",
    (final_keyword_coverage["kwic_hits"] > 0).sum()
)

print(
    "Terms with hits but not represented in validation sample:",
    (
        (final_keyword_coverage["kwic_hits"] > 0) &
        (~final_keyword_coverage["in_validation_sample"])
    ).sum()
)

print("\nTerms with zero full-corpus hits:")
display(
    final_keyword_coverage[
        final_keyword_coverage["kwic_hits"] == 0
    ]
)

In [ ]:
# Step 7: prepare and export manual validation file

validation_review = validation_sample_final.copy()

# Add manual review columns
validation_review["valid_mention"] = ""
validation_review["validity_note"] = ""

# Arrange columns for manual review
review_columns = [
    "transcript_id",
    "file_name",
    "category",
    "keyword",
    "matched_text",
    "context",
    "left_context",
    "right_context",
    "match_start",
    "match_end",
    "partial_transcript",
    "sample_source",
    "valid_mention",
    "validity_note"
]

validation_review = validation_review[review_columns]

# Sort for easier manual review
validation_review = validation_review.sort_values(
    ["category", "keyword", "transcript_id", "match_start"]
).reset_index(drop=True)

REVIEW_PATH = OUTPUT_DIR / "full_kwic_validation_sample_v2.xlsx"

validation_review.to_excel(
    REVIEW_PATH,
    index=False
)

print("Saved validation review file:")
print(REVIEW_PATH)

print("\nRows for manual review:", len(validation_review))

### Validation sample generated

A final validation sample of **275 single-KWIC records** was prepared for manual review.

The sample consists of:

- **250 category-balanced random records**: 50 from each of the five institution categories;
- **25 supplementary records** added to ensure coverage of dictionary terms not represented in the initial random sample.

All included dictionary terms with at least one full-corpus hit are represented in the validation sample. The only included term with zero full-corpus hits is `Womens Aid`.

Manual review at this stage assesses only:

- `valid_mention`
- `validity_note`

The results were used to evaluate full-corpus retrieval quality before TF-IDF and NMF analysis.

In [ ]:
# -----------------------------
# Step 8 — Load human-reviewed validation sample
# -----------------------------

import pandas as pd
from pathlib import Path

review_path = (
    OUTPUT_DIR /
    "full_kwic_validation_sample_v2_human_reviewed.xlsx"
)

validation_reviewed = pd.read_excel(review_path)

print("Shape:", validation_reviewed.shape)

print("\nColumns:")
print(validation_reviewed.columns.tolist())

print("\nHuman validity counts:")
print(
    validation_reviewed["valid_mention"]
    .value_counts(dropna=False)
)

In [ ]:
# -----------------------------
# Step 9 — Validate human labels
# -----------------------------

validation_reviewed["valid_mention"] = (
    validation_reviewed["valid_mention"]
    .astype("string")
    .str.strip()
    .str.lower()
)

allowed_labels = {"yes", "no", "unclear"}

missing_mask = validation_reviewed["valid_mention"].isna()

invalid_mask = (
    validation_reviewed["valid_mention"].notna()
    & ~validation_reviewed["valid_mention"].isin(allowed_labels)
)

print("Total rows:", len(validation_reviewed))
print("Missing human labels:", missing_mask.sum())
print("Invalid human labels:", invalid_mask.sum())

print("\nFinal human validity counts:")
print(
    validation_reviewed["valid_mention"]
    .value_counts(dropna=False)
)

In [ ]:
# -----------------------------
# Step 10 — Category-level validity summary
# -----------------------------

category_summary = (
    validation_reviewed
    .groupby("category")
    .agg(
        sampled_n=("valid_mention", "size"),
        valid_n=("valid_mention", lambda x: (x == "yes").sum()),
        invalid_n=("valid_mention", lambda x: (x == "no").sum())
    )
    .reset_index()
)

category_summary["validity_rate"] = (
    category_summary["valid_n"] / category_summary["sampled_n"]
)

category_summary["validity_pct"] = (
    category_summary["validity_rate"] * 100
).round(1)

category_summary = category_summary.sort_values(
    "validity_rate",
    ascending=False
)

display(category_summary)

In [ ]:
# -----------------------------
# Step 11 — Keyword-level validity summary
# -----------------------------

keyword_summary = (
    validation_reviewed
    .groupby(["category", "keyword"])
    .agg(
        sampled_n=("valid_mention", "size"),
        valid_n=("valid_mention", lambda x: (x == "yes").sum()),
        invalid_n=("valid_mention", lambda x: (x == "no").sum())
    )
    .reset_index()
)

keyword_summary["validity_rate"] = (
    keyword_summary["valid_n"] / keyword_summary["sampled_n"]
)

keyword_summary["validity_pct"] = (
    keyword_summary["validity_rate"] * 100
).round(1)

keyword_summary = keyword_summary.sort_values(
    ["category", "validity_rate", "sampled_n"],
    ascending=[True, True, False]
)

display(keyword_summary)

In [ ]:
# -----------------------------
# Step 12 — Diagnostic review of keyword-level precision
# -----------------------------

def precision_diagnostic(row):
    n = row["sampled_n"]
    rate = row["validity_rate"]

    # Too few sampled cases to interpret confidently
    if n < 3:
        return "insufficient_sample"

    # Descriptive diagnostic bands only
    elif rate < 0.40:
        return "low_precision"

    elif rate < 0.70:
        return "moderate_precision"

    else:
        return "high_precision"


keyword_summary["precision_diagnostic"] = keyword_summary.apply(
    precision_diagnostic,
    axis=1
)

diagnostic_order = {
    "low_precision": 0,
    "moderate_precision": 1,
    "insufficient_sample": 2,
    "high_precision": 3
}

keyword_summary["diagnostic_order"] = (
    keyword_summary["precision_diagnostic"].map(diagnostic_order)
)

keyword_diagnostic = (
    keyword_summary
    .sort_values(
        ["diagnostic_order", "validity_pct", "sampled_n"],
        ascending=[True, True, False]
    )
)

display(
    keyword_diagnostic[
        [
            "category",
            "keyword",
            "sampled_n",
            "valid_n",
            "invalid_n",
            "validity_pct",
            "precision_diagnostic"
        ]
    ]
)

### Interpretation

The diagnostic labels above are descriptive only and are not used as automatic exclusion rules.

Keyword-level validation is used to identify lexical ambiguity and recurring false-positive patterns. Decisions about the scope of subsequent substantive annotation are made at the institutional-category level, informed by both validation performance and analytical relevance.

In particular, the validation results showed substantially higher precision for legal, police and support-sector mentions than for health and welfare/state mentions. Subsequent substantive annotation therefore focuses on the three more reliable institutional domains, while health and welfare/state remain part of the exploratory retrieval and validation stage.

## Scope decision following validation

The initial retrieval strategy covered five broad institutional domains: police, legal, support sector, health, and welfare/state. Human validation showed substantial differences in retrieval precision across these domains.

Legal (92.5%), police (86.8%), and support-sector (79.3%) mentions showed comparatively high validation precision, while health (48.2%) and welfare/state (43.6%) produced substantially more false-positive and contextually ambiguous matches.

On the basis of these validation results, together with the need to maintain a feasible and analytically coherent scope, subsequent substantive annotation is focused on three institutional domains: police, legal institutions, and support-sector institutions.

Health and welfare/state are not removed from the exploratory stage of the study. Their retrieval and validation results are retained as part of the methodological assessment, but they are excluded from the main substantive annotation stage. This distinction preserves the broader exploratory search while concentrating detailed interpretive analysis on the institutional domains for which keyword-assisted retrieval proved more reliable.

In [ ]:
# -----------------------------
# Step 13 — Export validation summaries
# -----------------------------

from pathlib import Path
import pandas as pd

output_path = OUTPUT_DIR / "validation_summary_v2.xlsx"

# Remove helper sorting column before export
keyword_export = keyword_summary.drop(
    columns=["diagnostic_order"],
    errors="ignore"
).copy()

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:

    # Category-level validation results
    category_summary.to_excel(
        writer,
        sheet_name="category_summary",
        index=False
    )

    # Keyword-level validation results
    keyword_export.to_excel(
        writer,
        sheet_name="keyword_summary",
        index=False
    )

print("Validation summaries exported successfully.")
print(f"Saved to: {output_path}")

In [ ]:
# -----------------------------
# Step 14 — Validation performance for retained focus categories
# -----------------------------

focus_categories = [
    "police",
    "legal",
    "support_sector"
]

focus_validation = validation_reviewed[
    validation_reviewed["category"].isin(focus_categories)
].copy()

focus_summary = (
    focus_validation
    .groupby("category")
    .agg(
        sampled_n=("valid_mention", "size"),
        valid_n=("valid_mention", lambda x: (x == "yes").sum()),
        invalid_n=("valid_mention", lambda x: (x == "no").sum())
    )
    .reset_index()
)

focus_summary["validity_rate"] = (
    focus_summary["valid_n"] / focus_summary["sampled_n"]
)

focus_summary["validity_pct"] = (
    focus_summary["validity_rate"] * 100
).round(1)

display(focus_summary)

focus_total_n = len(focus_validation)
focus_valid_n = (focus_validation["valid_mention"] == "yes").sum()
focus_invalid_n = (focus_validation["valid_mention"] == "no").sum()
focus_validity_pct = focus_valid_n / focus_total_n * 100

print(f"Focus-category sample size: {focus_total_n}")
print(f"Valid mentions: {focus_valid_n}")
print(f"Invalid mentions: {focus_invalid_n}")
print(f"Combined validity: {focus_validity_pct:.1f}%")

In [ ]:
# -----------------------------
# Step 15 — Export focus-category validation summary
# -----------------------------

focus_output_path = (
    OUTPUT_DIR /
    "focus_category_validation_summary_v2.xlsx"
)

focus_summary.to_excel(
    focus_output_path,
    index=False
)

print("Focus-category validation summary exported successfully.")
print(f"Saved to: {focus_output_path}")

## Final scope decision

The validation stage began with five exploratory institutional categories. Human review showed clear differences in retrieval precision across these domains, with substantially stronger performance for legal (92.5%), police (86.8%), and support-sector (79.3%) mentions than for health (48.2%) and welfare/state (43.6%).

Subsequent substantive annotation therefore focused on **legal, police, and support-sector institutions**. Across these three retained categories, 141 of 164 validation cases were judged valid, giving a combined validation precision of **86.0%**.

Health and welfare/state are retained as part of the exploratory retrieval and validation record rather than treated as main substantive annotation categories. This decision reflects both their greater lexical ambiguity and the need to maintain a focused and feasible interpretive analysis.